# Imports

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(188)

# Download dataset

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# Understanding Grouped-Query Attention (GQA)

In the previous notebooks we used **Multi-Head Attention (MHA)** where every query head has its own dedicated Key and Value projections. This works well but becomes a memory bottleneck at scale — the KV cache grows linearly with the number of heads.

**Grouped-Query Attention (GQA)** is the solution used by Llama 2 70B, Llama 3, Mistral, and Gemma. It sits between two extremes:

| Variant | Q heads | KV heads | KV cache size |
|---------|---------|----------|---------------|
| **MHA** (Multi-Head Attention) | `n_head` | `n_head` | Full |
| **GQA** (Grouped-Query Attention) | `n_head` | `n_kv_head` | Reduced by `n_head / n_kv_head` |
| **MQA** (Multi-Query Attention) | `n_head` | 1 | Minimal |

### How it works

With `n_head=6` query heads and `n_kv_head=2` KV heads, every **3 query heads share 1 KV head**:

```
Query heads:   [Q0] [Q1] [Q2]   [Q3] [Q4] [Q5]
                 \   |   /         \   |   /
KV heads:        [K0, V0]          [K1, V1]
```

### Why it matters

- **KV cache memory** drops by `n_head / n_kv_head` (3x in our case)
- **Fewer parameters** in K/V projections
- **Nearly matches MHA quality** — unlike MQA which can degrade
- At scale (billions of params), this is the difference between fitting in GPU memory or not

### Implementation change

In our previous notebooks, attention used separate `Head` modules in a `ModuleList`. For GQA, we replace this with a single `GroupedQueryAttention` module that:
1. Projects Q with `n_head` output heads, but K and V with only `n_kv_head` output heads
2. Uses `repeat_interleave` to expand KV heads to match Q heads before computing attention
3. Stores only the compact `n_kv_head` tensors in the KV cache

# Llama Model with Grouped-Query Attention

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 100
eval_interval = 10
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6        # number of query heads
n_kv_head = 2     # number of key/value heads (GQA: shared across groups of query heads)
n_layer = 6
dropout = 0.2
# ------------

assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"

torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def precompute_rope_params(head_dim, theta_base=10_000, block_size=4096):
    assert head_dim % 2 == 0, "Head dimension must be even"
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))
    positions = torch.arange(block_size)
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
    angles = torch.cat([angles, angles], dim=1)
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    return cos, sin

################################### NEW: 4D RoPE ##############################
# Previous notebooks applied RoPE to 3D tensors (B, T, head_dim) one head at a
# time. With GQA we process all heads in a single tensor, so RoPE now operates
# on 4D tensors (B, n_heads, T, head_dim) and broadcasts over the head dimension.
###############################################################################
def compute_rope(x, cos, sin, start_pos=0):
    # x: (B, n_heads, T, head_dim)
    seq_len = x.shape[2]
    head_dim = x.shape[3]
    assert head_dim % 2 == 0, "Head dimension must be even"

    x1 = x[..., :head_dim // 2]
    x2 = x[..., head_dim // 2:]

    cos = cos[start_pos:start_pos+seq_len, :].view(1, 1, seq_len, head_dim)
    sin = sin[start_pos:start_pos+seq_len, :].view(1, 1, seq_len, head_dim)

    rotated = torch.cat((-x2, x1), dim=-1)
    return (x * cos) + (rotated * sin)


class GroupedQueryAttention(nn.Module):
    """ Grouped-Query Attention: Q heads are grouped, each group shares one K/V head """

    def __init__(self, n_embd, n_head, n_kv_head):
        super().__init__()
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_group = n_head // n_kv_head
        self.head_size = n_embd // n_head

        ################################### GQA PROJECTIONS ########################
        # Q has n_head projections, but K and V only have n_kv_head projections
        self.q_proj = nn.Linear(n_embd, n_head * self.head_size, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_size, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_size, bias=False)
        ###########################################################################
        self.out_proj = nn.Linear(n_head * self.head_size, n_embd, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        cos, sin = precompute_rope_params(head_dim=self.head_size, block_size=block_size)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

        self.register_buffer("cache_k", None)
        self.register_buffer("cache_v", None)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, use_cache=False, start_pos=0):
        B, T, C = x.shape

        # Project Q, K, V
        q = self.q_proj(x)  # (B, T, n_head * head_size)
        k = self.k_proj(x)  # (B, T, n_kv_head * head_size)
        v = self.v_proj(x)  # (B, T, n_kv_head * head_size)

        # Reshape to multi-head format
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)     # (B, n_head, T, hs)
        k = k.view(B, T, self.n_kv_head, self.head_size).transpose(1, 2)  # (B, n_kv_head, T, hs)
        v = v.view(B, T, self.n_kv_head, self.head_size).transpose(1, 2)  # (B, n_kv_head, T, hs)

        # Apply RoPE to Q and K
        q = compute_rope(q, self.cos, self.sin, start_pos=start_pos)
        k = compute_rope(k, self.cos, self.sin, start_pos=start_pos)

        # KV Cache — stores only n_kv_head entries, not n_head
        if use_cache:
            if self.cache_k is None:
                self.cache_k = k
                self.cache_v = v
            else:
                self.cache_k = torch.cat([self.cache_k, k], dim=2)
                self.cache_v = torch.cat([self.cache_v, v], dim=2)
            k, v = self.cache_k, self.cache_v

        ################################### GQA KEY STEP ###########################
        # Expand KV heads to match query heads by repeating each KV head n_group times
        # (B, n_kv_head, T_k, hs) -> (B, n_head, T_k, hs)
        k = k.repeat_interleave(self.n_group, dim=1)
        v = v.repeat_interleave(self.n_group, dim=1)
        ###########################################################################

        # Compute attention scores
        T_q, T_k = q.shape[2], k.shape[2]
        wei = q @ k.transpose(-2, -1) * self.head_size ** -0.5  # (B, n_head, T_q, T_k)
        wei = wei.masked_fill(
            self.tril[T_k-T_q:T_k, :T_k].unsqueeze(0).unsqueeze(0) == 0,
            float('-inf')
        )
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Weighted aggregation
        out = wei @ v  # (B, n_head, T_q, hs)
        out = out.transpose(1, 2).contiguous().view(B, T_q, self.n_head * self.head_size)
        return self.out_proj(out)


class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.SiLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head, n_kv_head):
        super().__init__()
        self.sa = GroupedQueryAttention(n_embd, n_head, n_kv_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.RMSNorm(n_embd)
        self.ln2 = nn.RMSNorm(n_embd)

    def forward(self, x, use_cache=False, start_pos=0):
        x = x + self.sa(self.ln1(x), use_cache=use_cache, start_pos=start_pos)
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head, n_kv_head=n_kv_head) for _ in range(n_layer)])
        self.ln_f = nn.RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False):
        B, T = idx.shape

        if use_cache:
            first_attn = self.blocks[0].sa
            cached_len = 0 if first_attn.cache_k is None else first_attn.cache_k.shape[2]
            idx = idx[:, cached_len:]
            start_pos = cached_len
        else:
            start_pos = 0

        tok_emb = self.token_embedding_table(idx)
        x = tok_emb
        for block in self.blocks:
            x = block(x, use_cache=use_cache, start_pos=start_pos)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def clear_cache(self):
        for block in self.blocks:
            block.sa.cache_k = None
            block.sa.cache_v = None

    def generate(self, idx, max_new_tokens, use_cache=True):
        if use_cache:
            self.clear_cache()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond, use_cache=use_cache)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel()
m = model.to(device)

In [ ]:
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

## Parameter Comparison: MHA vs GQA

In [ ]:
head_size = n_embd // n_head

# MHA: each of n_head heads has its own Q, K, V projection
mha_kv_params = 2 * n_head * (n_embd * head_size)
# GQA: only n_kv_head K and V projections
gqa_kv_params = 2 * n_kv_head * (n_embd * head_size)

print(f"Config: n_head={n_head}, n_kv_head={n_kv_head}, head_size={head_size}")
print(f"  Query heads per KV group: {n_head // n_kv_head}")
print()
print(f"MHA K/V params per layer:  {mha_kv_params:>10,}")
print(f"GQA K/V params per layer:  {gqa_kv_params:>10,}")
print(f"K/V param reduction:       {(1 - gqa_kv_params/mha_kv_params)*100:.1f}%")
print()
print(f"MHA KV cache per token per layer: {2 * n_head * head_size} floats")
print(f"GQA KV cache per token per layer: {2 * n_kv_head * head_size} floats")
print(f"KV cache reduction:              {(1 - n_kv_head/n_head)*100:.1f}%")
print()

mha_total = sum(p.numel() for p in m.parameters()) + (mha_kv_params - gqa_kv_params) * n_layer
gqa_total = sum(p.numel() for p in m.parameters())
print(f"Estimated total model params with MHA: {mha_total/1e6:.2f}M")
print(f"Actual total model params with GQA:    {gqa_total/1e6:.2f}M")
print(f"Overall param reduction:               {(1 - gqa_total/mha_total)*100:.1f}%")

## Training

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

## Inference

In [ ]:
import time

context = torch.zeros((1, 1), dtype=torch.long, device=device)
NUM_TOKENS = 200

model.eval()

# Time WITHOUT cache
with torch.no_grad():
    t0 = time.time()
    out_no_cache = model.generate(context.clone(), max_new_tokens=NUM_TOKENS, use_cache=False)
    t1 = time.time()
    time_no_cache = t1 - t0

# Time WITH cache
with torch.no_grad():
    t0 = time.time()
    out_cache = model.generate(context.clone(), max_new_tokens=NUM_TOKENS, use_cache=True)
    t1 = time.time()
    time_cache = t1 - t0

print(f"Without cache: {time_no_cache:.2f}s  ({NUM_TOKENS/time_no_cache:.1f} tok/s)")
print(f"With cache:    {time_cache:.2f}s  ({NUM_TOKENS/time_cache:.1f} tok/s)")
print(f"Speedup: {time_no_cache/time_cache:.2f}x")
print()
print("--- Sample output (with cache) ---")
print(decode(out_cache[0].tolist()))